# StatCore MLP: weight-decay selection, reliability head and predictive uncertainty

[Open in Colab](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/statcore-mlp.ipynb)

**Paper topic:** StatCore Multilayer Perceptrons: NumPy Backpropagation, Weight-Decay Selection, Reliability Heads, and Predictive Uncertainty — `docs/latex/statcore-mlp.tex` ([repository](https://github.com/Jabualani/ML-Reliabilty/blob/main/docs/latex/statcore-mlp.tex)).
**Experiment script reproduced here:** `docs/latex/experiments/mlp_experiment.py` (reduced seed count so the notebook runs in a few minutes).

StatCore is a pure-NumPy/SciPy model family in `ml_reliability.statcore`. Every model has a `variant`
argument; each variant below is compared with the standard reference implementation.
**scikit-learn is used only as the baseline/reference** (and for loading public datasets), never inside StatCore.

## 1. Install (parameter cell)

> **Note:** StatCore has **not been pushed / published yet**. Until the repository is public, the
> `git+https` install below will fail. In that case use the **fallback cell**: zip the repository folder,
> upload the zip to Colab (left sidebar → Files, or the `files.upload()` helper) and `pip install` it.
> Run exactly one of the two cells.

In [ ]:
# @title Install StatCore (PARAMETER: change the source if needed)
# Option A - once the repository is published:
!pip install "git+https://github.com/Jabualani/ML-Reliabilty.git"

In [ ]:
# @title FALLBACK install from an uploaded zip of the repository (use if Option A fails)
# Zip the repo folder (ML-Reliabilty.zip), then uncomment and run:
# from google.colab import files
# files.upload()                       # pick ML-Reliabilty.zip
# !unzip -q -o ML-Reliabilty.zip -d repo
# !pip install ./repo/ML-Reliabilty     # adjust to the folder that contains pyproject.toml

In [ ]:
# @title Parameters and imports
SEEDS = (11, 23)   # the paper uses more seeds; fewer here to keep the run short

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## 2. Method summary

Standardised inputs $h_0=z$; hidden layers $h_l=\mathrm{ReLU}(h_{l-1}W_l+b_l)$ (He initialisation $W\sim\mathcal N(0,2/n_{\text{in}})$), output $o=h_LW_{L+1}+b_{L+1}$. **Inverted dropout** with rate $p$ is applied to hidden activations in training only for `uncertainty_head`: $h\leftarrow h\odot m/(1-p)$, $m\sim\mathrm{Bernoulli}(1-p)$.

**Losses.** Classification: softmax cross-entropy $\mathcal L=-\frac1n\sum_i\log\mathrm{softmax}(o_i)_{y_i}$ (gradient $(P-T)/n$). Regression (standardised $y$): $\mathcal L=\frac1{2n}\sum_i(o_i-y_i)^2$. Heteroscedastic regression (`uncertainty_head`): two outputs $(\mu,\ell)$ with $\ell=\log\sigma^2$ clipped to $[-6,6]$:
$$\mathcal L=\frac1n\sum_i\tfrac12\Big(\ell_i+\frac{(y_i-\mu_i)^2}{e^{\ell_i}}\Big).$$

**Optimiser (AdamW-style).** Mini-batches (`batch_size`), $m\leftarrow0.9m+0.1g$, $v\leftarrow0.999v+0.001g^2$, bias-corrected $\hat m,\hat v$:
$$W\leftarrow W-\eta\,\frac{\hat m}{\sqrt{\hat v}+10^{-8}}-\eta\,\lambda\,W,$$
decoupled weight decay $\lambda$ on weight matrices only (not biases). A `validation_fraction`=15 % split drives early stopping: stop after `patience` epochs without improving the validation loss by $>10^{-5}$ and restore the best weights. Reference: sklearn `Pipeline(StandardScaler, MLP*(hidden=(32,32), alpha=1e-2, early_stopping=True,...))`.

**`base`** — fixed $\lambda$.

**`stat_regularised`** — trains one network per $\lambda\in$ `wd_grid` ($10^{-4}..1$) and keeps the one with the lowest validation loss (`wd_scores_`).

**`reliability_head`** — `head_fraction`=20 % of the data is held out from network training. A small logistic regression ($\alpha=1$) is trained on features of the held-out points to predict *whether the network is right*: $\|h_L\|$, the input energy $\overline{z^2}/\overline{z^2}_{\text{train}}$, and for classification $(\max p,\ H(p),\ p_{(1)}-p_{(2)})$, for regression $|\hat y|$. Target: correct prediction (classification) or $|{\rm error}|\le$ median error (regression). `predict_reliability(X)` returns the head's probability.

**`uncertainty_head`** — Monte-Carlo dropout with $S=$ `mc_samples` stochastic passes. Classification: $\bar p=\frac1S\sum_s\mathrm{softmax}(o_s)$, uncertainty $=H(\bar p)/\log C$. Regression: $\hat y=\frac1S\sum_s\mu_s$ and
$$\hat\sigma^2_{\text{pred}}=\underbrace{\frac1S\sum_s e^{\ell_s}}_{\text{aleatoric}}+\underbrace{\mathrm{Var}_s(\mu_s)}_{\text{epistemic}}\quad(\times\,\sigma_y^2\ \text{to original units}).$$

## 3. Experiment (mirrors `mlp_experiment.py` / `_experiment_utils.py`)

Same data protocol as the k-NN notebook. Networks: hidden $(32,32)$, lr $10^{-2}$, weight decay $10^{-2}$, batch 64, at most 100 epochs, patience 12, dropout 0.1, 20 MC samples, 20 % head split.

In [ ]:
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def _synthetic(kind, seed, n=640):
    rng = np.random.default_rng(seed + (10000 if kind == "regression" else 0))
    x = rng.uniform(-2.0, 2.0, size=(n, 2)); noise = rng.normal(size=n)
    if kind == "classification":
        y = ((x[:, 0] ** 2 + 0.65 * x[:, 1] ** 2 + 0.25 * noise) > 1.65).astype(int)
        flip = rng.random(n) < 0.10; y[flip] = 1 - y[flip]
    else:
        y = np.sin(1.4 * x[:, 0]) + 0.35 * x[:, 1] ** 2 + 0.30 * noise
    return np.column_stack((x, rng.normal(size=(n, 18)))), y      # 2 relevant + 18 irrelevant features

def datasets(kind, seed):
    X, y = _synthetic(kind, seed)
    yield "synthetic_nonlinear_irrelevant", X, y
    bunch = load_breast_cancer() if kind == "classification" else load_diabetes()
    yield ("sklearn_breast_cancer" if kind == "classification" else "sklearn_diabetes"), bunch.data, bunch.target

def split_data(X, y, seed, stratify):
    rng = np.random.default_rng(seed); n_test = int(np.ceil(0.25 * len(y)))
    if not stratify:
        order = rng.permutation(len(y))
        return X[order[n_test:]], X[order[:n_test]], y[order[n_test:]], y[order[:n_test]]
    classes, inverse, counts = np.unique(y, return_inverse=True, return_counts=True)
    raw = counts * (n_test / len(y)); allocation = np.floor(raw).astype(int)
    remaining = n_test - int(allocation.sum()); tie_break = rng.random(len(counts))
    order = np.lexsort((tie_break, -(raw - allocation))); allocation[order[:remaining]] += 1
    train_ids, test_ids = [], []
    for c, count in enumerate(allocation):
        ids = rng.permutation(np.flatnonzero(inverse == c)); test_ids.extend(ids[:count]); train_ids.extend(ids[count:])
    train_ids = rng.permutation(np.asarray(train_ids, dtype=int)); test_ids = rng.permutation(np.asarray(test_ids, dtype=int))
    return X[train_ids], X[test_ids], y[train_ids], y[test_ids]

def ece_equal_width(y, prob, bins=15):
    pred = prob.argmax(axis=1); conf = prob.max(axis=1); correct = (pred == y).astype(float)
    edges = np.linspace(0.0, 1.0, bins + 1); ans = 0.0
    for j in range(bins):
        mask = (conf >= edges[j]) & ((conf < edges[j + 1]) if j < bins - 1 else (conf <= edges[j + 1]))
        if mask.any():
            ans += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
    return float(ans)

def classification_scores(y, prob):
    prob = np.clip(np.asarray(prob), 1e-12, 1.0); prob = prob / prob.sum(axis=1, keepdims=True)
    truth = np.searchsorted(np.unique(y), y)
    return {"accuracy": float(np.mean(truth == prob.argmax(axis=1))),
            "log_loss": float(-np.log(prob[np.arange(len(y)), truth]).mean()),
            "ece": ece_equal_width(truth, prob)}

def regression_scores(y, pred):
    residual = np.asarray(y) - np.asarray(pred); total = np.sum((y - np.mean(y)) ** 2)
    return {"mse": float(np.mean(residual ** 2)), "mae": float(np.mean(np.abs(residual))),
            "r2": float(1.0 - np.sum(residual ** 2) / total)}

def run_family(make_models, seeds):
    import time
    rows = []
    for task in ("classification", "regression"):
        for seed in seeds:
            for dataset, X, y in datasets(task, seed):
                Xtr, Xte, ytr, yte = split_data(X, y, seed=seed, stratify=(task == "classification"))
                for name, model in make_models(task, seed).items():
                    start = time.perf_counter(); model.fit(Xtr, ytr)
                    if task == "classification":
                        prob = model.predict_proba(Xte); classes = model.classes_
                        if not np.array_equal(classes, np.arange(len(classes))):
                            prob = prob[:, [int(np.flatnonzero(classes == c)[0]) for c in np.arange(len(classes))]]
                        metrics = classification_scores(yte, prob)
                    else:
                        metrics = regression_scores(yte, model.predict(Xte))
                    rows.append({"task": task, "dataset": dataset, "seed": seed, "model": name,
                                 "runtime_s": time.perf_counter() - start, **metrics})
    return pd.DataFrame(rows)

def summarise(res, order):
    for task, metrics in (("classification", ["accuracy", "log_loss", "ece", "runtime_s"]),
                          ("regression", ["mse", "mae", "r2", "runtime_s"])):
        sub = res[res.task == task]
        g = sub.groupby(["dataset", "model"], sort=False)[metrics]
        m, s = g.mean(), g.std()
        out = m.round(4).astype(str) + " +/- " + s.round(4).astype(str)
        print(f"\n[{task}] mean +/- sample SD over seeds"); print(out.loc[pd.MultiIndex.from_product([sub.dataset.unique(), order])].to_string())
    return res

from sklearn.neural_network import MLPClassifier as SklearnMLPClassifier         # baseline only
from sklearn.neural_network import MLPRegressor as SklearnMLPRegressor           # baseline only
from ml_reliability.statcore import MLPClassifier, MLPRegressor

VARIANT_NAMES = ("base", "stat_regularised", "reliability_head", "uncertainty_head")

def make_models(task, seed):
    is_class = task == "classification"
    sk_cls = SklearnMLPClassifier if is_class else SklearnMLPRegressor
    baseline = make_pipeline(StandardScaler(), sk_cls(hidden_layer_sizes=(32, 32), alpha=1e-2, batch_size=64,
                             learning_rate_init=1e-2, max_iter=100, early_stopping=True, n_iter_no_change=12, random_state=seed))
    cls = MLPClassifier if is_class else MLPRegressor
    models = {name: cls(hidden_layer_sizes=(32, 32), variant=name, learning_rate=1e-2, weight_decay=1e-2,
                        max_epochs=100, batch_size=64, patience=12, validation_fraction=0.15, dropout=0.1,
                        mc_samples=20, head_fraction=0.2, random_state=seed) for name in VARIANT_NAMES}
    return {"sklearn_standard": baseline, **models}

ORDER = ["sklearn_standard", *VARIANT_NAMES]
res = run_family(make_models, SEEDS)
print(f"MLP experiment over seeds {SEEDS} (runtime = single fit + predict, no warm-up)")
res = summarise(res, ORDER)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, (task, metric) in zip(axes, (("classification", "log_loss"), ("regression", "mse"))):
    t = res[res.task == task].groupby(["dataset", "model"])[metric].mean().unstack("model")[ORDER]
    t.plot.bar(ax=ax, rot=15, legend=(task == "regression")); ax.set_title(f"{task}: {metric}")
    if metric == "mse": ax.set_yscale("log")
plt.tight_layout(); plt.show()

## 4. Reliability head, uncertainty head and weight-decay selection

Do the reliability / uncertainty scores actually separate good predictions from bad ones? We check on the held-out breast-cancer and diabetes splits.

In [ ]:
def auroc(score, positive):
    # probability that a random positive outscores a random negative (ties count 1/2)
    pos, neg = score[positive], score[~positive]
    if len(pos) == 0 or len(neg) == 0: return float("nan")
    return float(((pos[:, None] > neg[None, :]).mean() + 0.5 * (pos[:, None] == neg[None, :]).mean()))

seed = SEEDS[0]
Xtr, Xte, ytr, yte = split_data(load_breast_cancer().data, load_breast_cancer().target, seed, True)
kw = dict(hidden_layer_sizes=(32, 32), learning_rate=1e-2, weight_decay=1e-2, max_epochs=100, batch_size=64,
          patience=12, dropout=0.1, mc_samples=20, head_fraction=0.2, random_state=seed)
rows = []
for variant in ("base", "reliability_head", "uncertainty_head"):
    m = MLPClassifier(variant=variant, **kw).fit(Xtr, ytr)
    correct = m.predict(Xte) == yte
    rows.append(dict(task="classification", variant=variant, accuracy=correct.mean(),
                     AUROC_reliability_vs_correct=auroc(m.reliability_score(Xte), correct)))
Xtr, Xte, ytr, yte = split_data(load_diabetes().data, load_diabetes().target, seed, False)
for variant in ("reliability_head", "uncertainty_head"):
    m = MLPRegressor(variant=variant, **kw).fit(Xtr, ytr)
    err = np.abs(m.predict(Xte) - yte)
    score = m.reliability_score(Xte)
    rows.append(dict(task="regression", variant=variant, accuracy=np.nan,
                     AUROC_reliability_vs_correct=auroc(score, err <= np.median(err))))
    if variant == "uncertainty_head":
        sd = m.predict_uncertainty(Xte)
        lo, hi = m.predict(Xte) - 1.645 * sd, m.predict(Xte) + 1.645 * sd
        rows[-1]["coverage90_gaussian"] = float(np.mean((yte >= lo) & (yte <= hi)))
        rows[-1]["corr(sd,|err|)"] = float(np.corrcoef(sd, err)[0, 1])
print("AUROC 0.5 = uninformative, 1.0 = perfectly ranks trustworthy predictions first")
pd.DataFrame(rows).round(3)

In [ ]:
sr = MLPClassifier(variant="stat_regularised", **{k: v for k, v in kw.items() if k != "weight_decay"}).fit(*split_data(*_synthetic("classification", seed), seed, True)[::2])
ws = sorted(sr.wd_scores_)
plt.figure(figsize=(4.8, 3.4)); plt.semilogx(ws, [sr.wd_scores_[w] for w in ws], "o-"); plt.axvline(sr.weight_decay_, color="r", ls="--")
plt.xlabel("weight decay"); plt.ylabel("validation cross-entropy"); plt.title(f"stat_regularised: chosen {sr.weight_decay_:g}"); plt.show()

## 5. Summary and limitations

In [ ]:
for task, metric, smaller in (("classification", "log_loss", True), ("regression", "mse", True)):
    t = res[res.task == task].groupby(["dataset", "model"])[metric].mean().unstack("model")[ORDER]
    print(f"best {metric} per dataset:", (t.idxmin(axis=1) if smaller else t.idxmax(axis=1)).to_dict())

**What to read from the tables.** In our local 2-seed run `stat_regularised` and `uncertainty_head` gave the best regression MSE on the synthetic set (0.23 and 0.34 vs 0.40 for sklearn), whereas on the synthetic classification set the sklearn MLP had the highest accuracy. The heads are meant to *rank* predictions by trustworthiness, so judge them in the AUROC table: on breast cancer the MC-dropout score (`uncertainty_head`, AUROC ≈ 0.95) separated right from wrong predictions about as well as plain max-probability (`base`, ≈ 0.96), whereas the `reliability_head` score was **anti-informative** in that run (AUROC ≈ 0.17; the head was fitted on ~85 rows with only a handful of errors).

**Honest limitations**
- Two seeds (3 in the paper), small datasets, single architecture; differences between variants are mostly within one SD. Runtime is an un-warmed single fit.
- `reliability_head` removes 20 % of the data from network training and its head is trained on few points (~85 for breast cancer after the split, with very few errors to learn from): the AUROC is noisy, it can be below 0.5 (as above) and the head is near-constant when predictions are almost all correct. Test on one seed only here.
- MC dropout approximates epistemic uncertainty only roughly, is not calibrated by construction, and the Gaussian 90 % coverage above is a diagnostic, not a guarantee. The regression head assumes Gaussian noise.
- `stat_regularised` trains five networks (5x cost) and picks $\lambda$ on a single validation split, which is itself noisy; early stopping already regularises, so the selected $\lambda$ often matters little.
- The pure-NumPy training loop is single-threaded and slower than optimised libraries for large data; no GPU, no convolutional/recurrent layers.